In [18]:
from store import VectorStore

store = VectorStore()
col = store.collection
print(col.count())  # expect 463

463


In [2]:
def show(ids):
    r = col.get(ids=ids, include=["documents", "metadatas"])
    for i, d, m in zip(r["ids"], r["documents"], r["metadatas"]):
        print("=" * 80)
        print(i, "\n", m)
        print("-" * 40)
        print(d[:500])

# a normal article, a split article, an annex, a schedule
show(["art-19", "art-175A-p1", "annex-objectives-resolution-p1", "first-schedule-p1"])

# an omitted article (filter, since I don't know its exact ID)
r = col.get(where={"article_number": "184"}, include=["documents", "metadatas"])
print(r["ids"], r["metadatas"], r["documents"][0][:300])

art-19 
 {'chunk_count': 1, 'chunk_index': 0, 'kind': 'article', 'article_num': 19, 'chapter': 'CHAPTER 1 – FUNDAMENTAL RIGHTS', 'pdf_page': 35, 'amendment_notes': '1. Subs. by the Constitution (Fourth Amdt.) Act, 1975 (71 of 1975), s. 4, for ‘‘defamation’’ (w.e.f. the 21st November, 1975). | 2. New Article 19A ins. by the Constitution (Eighteenth Amdt.) Act, 2010 (10 of 2010), s. 7.', 'printed_page': '13', 'article_number': '19', 'doc': 'Constitution of Pakistan', 'article_title': 'Freedom of speech, etc.', 'part': 'PART II – Fundamental Rights and Principles of Policy'}
----------------------------------------
Constitution of Pakistan | PART II – Fundamental Rights and Principles of Policy | CHAPTER 1 – FUNDAMENTAL RIGHTS | Article 19: Freedom of speech, etc.

19. Every citizen shall have the right to freedom of speech and expression, and there shall be freedom of the press, subject to any reasonable restrictions imposed by law in the interest of the glory of Islam or the integrity, 

In [3]:
PREFIX = "Represent this sentence for searching relevant passages: "

def run(q, n=10, prefix=False):
    res = store.query((PREFIX + q) if prefix else q, n_results=n)
    print(f"\nQ: {q}  [prefix={prefix}]")
    for id_, meta, dist in zip(res["ids"][0], res["metadatas"][0], res["distances"][0]):
        label = meta.get("article_number") or meta.get("section") or meta.get("kind")
        print(f"  {dist:.3f}  {id_:35s} {label}")

for q in ["freedom of speech and the press", "who appoints the Chief Justice"]:
    run(q, prefix=False)
    run(q, prefix=True)


Q: freedom of speech and the press  [prefix=False]
  0.220  art-19                              19
  0.315  art-66-p1                           66
  0.332  art-19A                             19A
  0.339  art-28                              28
  0.360  art-57                              57
  0.374  art-26                              26
  0.378  art-16                              16
  0.382  art-159-p1                          159
  0.383  art-226                             226
  0.385  art-17                              17

Q: freedom of speech and the press  [prefix=True]
  0.225  art-19                              19
  0.320  art-66-p1                           66
  0.343  art-28                              28
  0.358  art-19A                             19A
  0.369  art-57                              57
  0.382  art-26                              26
  0.383  art-16                              16
  0.394  art-17                              17
  0.397  art-226             

In [4]:
import re
from dataclasses import dataclass

PREFIX = "Represent this sentence for searching relevant passages: "
INC = ["documents", "metadatas"]
NUM = r"\d{1,3}[A-Za-z]{0,2}"
ART_RE = re.compile(
    rf"\b(?:articles?|arts?\.?)\s+({NUM}(?:\s*(?:,|and|&|to|through|-|–)\s*{NUM})*)", re.I)
MAX_RANGE = 20  # ignore silly ranges like "articles 1 to 280"

def parse_articles(q):
    """-> (list of article_number strings, list of (lo, hi) int ranges)"""
    nums, ranges = [], []
    for m in ART_RE.finditer(q):
        span = m.group(1)
        toks = [t.upper() for t in re.findall(NUM, span)]
        is_range = re.search(r"\bto\b|\bthrough\b|[-–]", span, re.I) and len(toks) == 2
        if is_range:
            lo, hi = (int(re.match(r"\d+", t).group()) for t in toks)
            if 0 <= hi - lo <= MAX_RANGE:
                ranges.append((lo, hi))
                continue
        nums += [t for t in toks if t not in nums]
    return nums, ranges

@dataclass
class Hit:
    key: str            # "19", "175A", "Preamble", "First Schedule", ...
    kind: str
    title: str
    text: str           # all parts, in order
    printed_pages: list
    pdf_pages: list
    amendment_notes: str  # page-level: may include neighbouring articles
    omitted: bool
    distance: float | None
    source: str         # "exact" | "semantic"

def _key(meta):
    return meta.get("article_number") or meta.get("section") or meta.get("kind")

def _build_hit(key, got, distance, source):
    rows = sorted(zip(got["documents"], got["metadatas"]),
                  key=lambda r: r[1].get("chunk_index", 0))
    metas = [m for _, m in rows]
    notes = []
    for m in metas:
        n = m.get("amendment_notes")
        if n and n not in notes:
            notes.append(n)
    first = metas[0]
    return Hit(
        key=key, kind=first.get("kind"),
        title=first.get("article_title") or first.get("section") or first.get("kind"),
        text="\n\n".join(d for d, _ in rows),
        printed_pages=sorted({m["printed_page"] for m in metas if m.get("printed_page")}),
        pdf_pages=sorted({m["pdf_page"] for m in metas if "pdf_page" in m}),
        amendment_notes=" || ".join(notes),
        omitted=any(m.get("omitted") for m in metas),
        distance=distance, source=source)

def _fetch_group(meta):
    where = ({"article_number": meta["article_number"]} if "article_number" in meta
             else {"section": meta["section"]})
    return col.get(where=where, include=INC)

def exact_hits(nums, ranges):
    hits = []
    for n in nums:
        got = col.get(where={"article_number": n}, include=INC)
        if got["ids"]:
            hits.append(_build_hit(n, got, None, "exact"))
    for lo, hi in ranges:
        got = col.get(where={"$and": [{"article_num": {"$gte": lo}},
                                      {"article_num": {"$lte": hi}}]}, include=INC)
        buckets = {}
        for d, m in zip(got["documents"], got["metadatas"]):
            buckets.setdefault(m["article_number"], {"documents": [], "metadatas": []})
            buckets[m["article_number"]]["documents"].append(d)
            buckets[m["article_number"]]["metadatas"].append(m)
        for k in sorted(buckets, key=lambda k: (buckets[k]["metadatas"][0]["article_num"], k)):
            hits.append(_build_hit(k, buckets[k], None, "exact"))
    return hits

def rank_keys(q, n_fetch=30, prefix=True, where=None):
    """Distinct articles/sections ranked by best chunk distance (no sibling fetch, cheap for eval)."""
    res = store.query((PREFIX + q) if prefix else q, n_results=n_fetch, where=where)
    best = {}
    for meta, d in zip(res["metadatas"][0], res["distances"][0]):
        best.setdefault(_key(meta), (d, meta))  # results arrive sorted, first = best
    return [(k, d, m) for k, (d, m) in best.items()]

def search(q, n_fetch=30, n_articles=5, prefix=True):
    nums, ranges = parse_articles(q)
    hits = exact_hits(nums, ranges)
    seen = {h.key for h in hits}
    sem = 0
    for k, d, meta in rank_keys(q, n_fetch, prefix):
        if k in seen:
            continue
        hits.append(_build_hit(k, _fetch_group(meta), d, "semantic"))
        seen.add(k)
        sem += 1
        if sem == n_articles:
            break
    return hits

# quick look
for h in search("How does Article 19 relate to defamation?"):
    print(h.source, h.key, h.title[:50], h.printed_pages, h.omitted, len(h.text))
print(parse_articles("Compare Articles 10 to 12 and Art. 175A"))

exact 19 Freedom of speech, etc. ['13'] False 587
semantic 269 Validation of laws, acts, etc. ['171'] False 1983
semantic 203D Powers, jurisdiction and functions of the court. ['118', '119'] False 3605
semantic 204 Contempt of Court. ['124'] False 892
semantic 174 Suits and pro-ceedings. ['89'] False 298
semantic 185 Appellate Jurisdiction of the Supreme Court. ['104'] False 2210
(['175A'], [(10, 12)])


In [5]:
GOLD = [
    ("freedom of speech and the press",                 {"19"}),
    ("minimum age to become President",                 {"41"}),
    ("who appoints the Chief Justice",                  {"175A", "177"}),
    ("can someone be detained without trial",           {"10"}),
    ("right to a fair trial",                           {"10A"}),
    ("right to information",                            {"19A"}),
    ("right to free education for children",            {"25A"}),
    ("are all citizens equal before the law",           {"25"}),
    ("protection of property rights",                   {"24"}),
    ("freedom to practise religion",                    {"20"}),
    ("qualifications to be a member of parliament",     {"62"}),
    ("disqualification of members of parliament",       {"63", "63A"}),
    ("how can the President be impeached",              {"47"}),
    ("proclamation of emergency because of war",        {"232"}),
    ("how is the Constitution amended",                 {"239"}),
    ("what is the national language",                   {"251"}),
    ("Council of Common Interests",                     {"153"}),
    ("who is the Chief Election Commissioner",          {"213"}),
    ("High Court writ jurisdiction habeas corpus",      {"199"}),
    ("local government system",                         {"140A"}),
    ("what is the Objectives Resolution",               {"Annex (Objectives Resolution)", "2A"}),
]

def evaluate(prefix, n_fetch=30, verbose=False):
    ks = (1, 3, 5)
    hit = {k: 0 for k in ks}; rr = 0.0; misses = []
    for q, gold in GOLD:
        keys = [k for k, _, _ in rank_keys(q, n_fetch, prefix)]
        rank = next((i + 1 for i, k in enumerate(keys) if k in gold), None)
        for k in ks:
            hit[k] += bool(rank and rank <= k)
        rr += 1 / rank if rank else 0
        if not rank or rank > 3:
            misses.append((q, gold, rank, keys[:5]))
    n = len(GOLD)
    print(f"prefix={prefix}  hit@1={hit[1]/n:.2f}  hit@3={hit[3]/n:.2f}  "
          f"hit@5={hit[5]/n:.2f}  MRR={rr/n:.2f}")
    if verbose:
        for m in misses: print("   miss:", m)

evaluate(False, verbose=True)
evaluate(True, verbose=True)

prefix=False  hit@1=0.95  hit@3=1.00  hit@5=1.00  MRR=0.98
prefix=True  hit@1=0.95  hit@3=1.00  hit@5=1.00  MRR=0.98


In [7]:
HARD = [
    ("can the government shut down newspapers",                 {"19"}),
    ("what happens if the PM loses majority support",           {"95"}),   # no-confidence; check
    ("who commands the armed forces",                           {"243"}),
    ("can police arrest me without telling me why",             {"10"}),
    ("is untouchability allowed",                               {"11"}),
    ("can a person be forced to work",                          {"11"}),
    ("how long can parliament sit before dissolution",          {"52"}),   # check
    ("can a judge be removed from office",                      {"209", "209A"}),  # 209 may be altered by 26th/27th Amdt.; verify
    ("who can declare someone a non-Muslim",                    {"260"}),
    ("can the army take over civil administration",             {"245"}),
    ("what are the provinces of Pakistan",                      {"1"}),
    ("rights of minorities",                                    {"36"}),
]

GOLD_ALL = GOLD + HARD

def evaluate2(gold, prefix, n_fetch=30):
    ks = (1, 3, 5); hit = {k: 0 for k in ks}; rr = 0.0
    for q, g in gold:
        keys = [k for k, _, _ in rank_keys(q, n_fetch, prefix)]
        rank = next((i + 1 for i, k in enumerate(keys) if k in g), None)
        for k in ks: hit[k] += bool(rank and rank <= k)
        rr += 1 / rank if rank else 0
        if not rank or rank > 3: print("  miss:", q, g, rank, keys[:5])
    n = len(gold)
    print(f"prefix={prefix} hit@1={hit[1]/n:.2f} hit@3={hit[3]/n:.2f} hit@5={hit[5]/n:.2f} MRR={rr/n:.2f}")

evaluate2(HARD, False)
evaluate2(HARD, True)

  miss: is untouchability allowed {'11'} None ['159', '211', '269', '203G', '69']
prefix=False hit@1=0.67 hit@3=0.92 hit@5=0.92 MRR=0.78
  miss: is untouchability allowed {'11'} None ['159', '211', '69', '237', '203G']
prefix=True hit@1=0.75 hit@3=0.92 hit@5=0.92 MRR=0.83


In [8]:
for q in ["How does Article 19 relate to defamation?", "can the government shut down newspapers"]:
    print("\n", q)
    for k, d, _ in rank_keys(q, 30, False)[:8]:
        print(f"  {d:.3f}  {k}")


 How does Article 19 relate to defamation?
  0.310  269
  0.327  19
  0.327  203D
  0.345  155
  0.346  185
  0.346  267B
  0.346  174
  0.349  66

 can the government shut down newspapers
  0.380  159
  0.388  19
  0.399  89
  0.401  267B
  0.406  224
  0.410  Second Schedule
  0.411  95
  0.413  216


In [9]:
from rank_bm25 import BM25Okapi

ALL = col.get(include=INC)                      # all 463 chunks
TOK = lambda s: re.findall(r"[a-z0-9]+", s.lower())
bm25 = BM25Okapi([TOK(d) for d in ALL["documents"]])

def bm25_keys(q, top=30):
    scores = bm25.get_scores(TOK(q))
    order = sorted(range(len(scores)), key=lambda i: -scores[i])[:top]
    best = {}
    for i in order:
        if scores[i] <= 0:
            break
        best.setdefault(_key(ALL["metadatas"][i]), scores[i])
    return list(best)

def rrf(rankings, k=60):
    sc = {}
    for r in rankings:
        for i, key in enumerate(r):
            sc[key] = sc.get(key, 0) + 1 / (k + i + 1)
    return sorted(sc, key=sc.get, reverse=True)

def sem_keys(q, n_fetch=30, prefix=True):
    return [k for k, _, _ in rank_keys(q, n_fetch, prefix)]

def hybrid_keys(q, n_fetch=30, prefix=True):
    return rrf([sem_keys(q, n_fetch, prefix), bm25_keys(q)])

def evaluate3(gold, keyfn, label):
    ks = (1, 3, 5); hit = {k: 0 for k in ks}; rr = 0.0
    for q, g in gold:
        keys = keyfn(q)
        rank = next((i + 1 for i, k in enumerate(keys) if k in g), None)
        for k in ks: hit[k] += bool(rank and rank <= k)
        rr += 1 / rank if rank else 0
        if not rank or rank > 1:
            print(f"   [{label}] rank={rank}  {q}  gold={g}  top5={keys[:5]}")
    n = len(gold)
    print(f"{label:8s} hit@1={hit[1]/n:.2f} hit@3={hit[3]/n:.2f} hit@5={hit[5]/n:.2f} MRR={rr/n:.2f}\n")

GOLD_ALL = GOLD + HARD
evaluate3(GOLD_ALL, sem_keys, "semantic")
evaluate3(GOLD_ALL, bm25_keys, "bm25")
evaluate3(GOLD_ALL, hybrid_keys, "hybrid")

# diagnostic: where does Article 11 land for the missed query?
q = "is untouchability allowed"
print("sem n_fetch=100:", sem_keys(q, 100).index("11") + 1 if "11" in sem_keys(q, 100) else "not found")
print("bm25 :", bm25_keys(q)[:5])
print("hybrid:", hybrid_keys(q)[:5])

   [semantic] rank=2  how is the Constitution amended  gold={'239'}  top5=['238', '239', '268', '266', '267B']
   [semantic] rank=2  can the government shut down newspapers  gold={'19'}  top5=['159', '19', '89', '267B', '91']
   [semantic] rank=None  is untouchability allowed  gold={'11'}  top5=['159', '211', '69', '237', '203G']
   [semantic] rank=2  can the army take over civil administration  gold={'245'}  top5=['256', '245', '145', '243', '130']
semantic hit@1=0.88 hit@3=0.97 hit@5=0.97 MRR=0.92

   [bm25] rank=11  minimum age to become President  gold={'41'}  top5=['Fifth Schedule', '37', '63', '25A', '224']
   [bm25] rank=3  who appoints the Chief Justice  gold={'177', '175A'}  top5=['209', '203C', '175A', '180', '275']
   [bm25] rank=4  qualifications to be a member of parliament  gold={'62'}  top5=['271', '113', '273', '62', '63']
   [bm25] rank=7  how is the Constitution amended  gold={'239'}  top5=['238', '71', '241', '152A', 'Fifth Schedule']
   [bm25] rank=5  High Court wri

In [10]:
r = col.get(where={"article_number": "11"}, include=["documents"])
print(r["documents"][0])

# how widespread are leftover hyphens inside words?
hy = [m.group() for d in ALL["documents"] for m in re.finditer(r"\b[a-z]{2,}-[a-z]{2,}\b", d)]
print(len(hy), "hyphenated tokens; sample:", hy[:40])

Constitution of Pakistan | PART II – Fundamental Rights and Principles of Policy | CHAPTER 1 – FUNDAMENTAL RIGHTS | Article 11: Slavery, forced labour, etc., prohibited.

11. (1) Slavery is non-existent and forbidden and no law shall permit or facilitate its introduction into Pakistan in any form.

(2) All forms of forced labour and traffic in human beings are prohibited.

(3) No child below the age of fourteen years shall be engaged in any factory or mine or any other hazardous employment.

(4) Nothing in this Article shall be deemed to affect compulsory service–

(a) by any person undergoing punishment for an offence against any law; or

(b) required by any law for public purpose:

Provided that no compulsory service shall be of a cruel nature or incompatible with human dignity.
278 hyphenated tokens; sample: ['commenc-ing', 'twenty-four', 'twenty-four', 'anti-national', 'anti-national', 'non-existent', 'twenty-third', 'fifty-six', 'self-incrimination', 'educa-tional', 'under-represe

In [16]:
STOP = set("""a an the is are was were be been of to in on for and or can could may might
will shall do does did i me my we you it its that this what who whom which how when where
why if as at by from with about into than then there their they them not no""".split())

def bm25_keys2(q, top=30):
    toks = [t for t in TOK(q) if t not in STOP]
    scores = bm25.get_scores(toks)
    order = sorted(range(len(scores)), key=lambda i: -scores[i])[:top]
    best = {}
    for i in order:
        if scores[i] <= 0: break
        best.setdefault(_key(ALL["metadatas"][i]), scores[i])
    return list(best)

def wrrf(rankings, weights, k=60):
    sc = {}
    for r, w in zip(rankings, weights):
        for i, key in enumerate(r):
            sc[key] = sc.get(key, 0) + w / (k + i + 1)
    return sorted(sc, key=sc.get, reverse=True)

def evaluate4(gold, keyfn, label):
    ks = (1, 3, 5); hit = {k: 0 for k in ks}; rr = 0.0
    for q, g in gold:
        keys = keyfn(q)
        rank = next((i + 1 for i, k in enumerate(keys) if k in g), None)
        for k in ks: hit[k] += bool(rank and rank <= k)
        rr += 1 / rank if rank else 0
    n = len(gold)
    print(f"{label:14s} hit@1={hit[1]/n:.2f} hit@3={hit[3]/n:.2f} hit@5={hit[5]/n:.2f} MRR={rr/n:.2f}")

evaluate4(GOLD_ALL, sem_keys, "semantic")
evaluate4(GOLD_ALL, bm25_keys2, "bm25+stop")
for w in (1.0, 0.5, 0.3):
    evaluate4(GOLD_ALL,
              lambda q, w=w: wrrf([sem_keys(q), bm25_keys2(q)], [1.0, w]),
              f"hybrid w={w}")

semantic       hit@1=0.91 hit@3=1.00 hit@5=1.00 MRR=0.95
bm25+stop      hit@1=0.59 hit@3=0.72 hit@5=0.81 MRR=0.67
hybrid w=1.0   hit@1=0.75 hit@3=0.88 hit@5=0.94 MRR=0.84
hybrid w=0.5   hit@1=0.81 hit@3=0.94 hit@5=0.94 MRR=0.88
hybrid w=0.3   hit@1=0.88 hit@3=0.94 hit@5=0.97 MRR=0.92


In [13]:
HARD = [x for x in HARD if "untouchability" not in x[0]]
GOLD_ALL = GOLD + HARD
NOANSWER = [   # for the RAG stage: bot should say "not addressed"
    "is untouchability allowed",
    "what is the minimum wage in Pakistan",
    "how do I get a driving licence",
]
text = " ".join(ALL["documents"]).lower()
words = set(re.findall(r"(?<![-\w])[a-z]{3,}(?![-\w])", text))
cands = {}
for m in re.finditer(r"\b([a-z]{2,})-([a-z]{2,})\b", text):
    a, b = m.groups()
    if a + b in words:
        cands[m.group()] = a + b
print(len(cands))
for k, v in sorted(cands.items()):
    print(f"{k:25s} -> {v}")

69
accor-dance               -> accordance
agricul-tural             -> agricultural
appoin-ted                -> appointed
authori-ties              -> authorities
cal-endar                 -> calendar
cer-tain                  -> certain
co-operative              -> cooperative
co-ordination             -> coordination
com-mencement             -> commencement
com-mission               -> commission
commenc-ing               -> commencing
connec-tion               -> connection
consider-ation            -> consideration
consti-tutional           -> constitutional
constitu-tion             -> constitution
construc-tion             -> construction
continu-ance              -> continuance
continu-ing               -> continuing
corpo-rations             -> corporations
deter-mined               -> determined
diffi-culties             -> difficulties
discrimi-nates            -> discriminates
disqualifi-cations        -> disqualifications
educa-tional              -> educational
exceed-i

In [14]:
text = " ".join(ALL["documents"]).lower()
words = set(re.findall(r"(?<![-\w])[a-z]{3,}(?![-\w])", text))
cands = {}
for m in re.finditer(r"\b([a-z]{2,})-([a-z]{2,})\b", text):
    a, b = m.groups()
    if a + b in words:
        cands[m.group()] = a + b
print(len(cands))
for k, v in sorted(cands.items()):
    print(f"{k:25s} -> {v}")

69
accor-dance               -> accordance
agricul-tural             -> agricultural
appoin-ted                -> appointed
authori-ties              -> authorities
cal-endar                 -> calendar
cer-tain                  -> certain
co-operative              -> cooperative
co-ordination             -> coordination
com-mencement             -> commencement
com-mission               -> commission
commenc-ing               -> commencing
connec-tion               -> connection
consider-ation            -> consideration
consti-tutional           -> constitutional
constitu-tion             -> constitution
construc-tion             -> construction
continu-ance              -> continuance
continu-ing               -> continuing
corpo-rations             -> corporations
deter-mined               -> determined
diffi-culties             -> difficulties
discrimi-nates            -> discriminates
disqualifi-cations        -> disqualifications
educa-tional              -> educational
exceed-i

In [15]:
import re

KEEP_HYPHEN = {"co-operative", "co-ordination", "up-to"}

def build_vocab(full_text):
    return set(re.findall(r"(?<![-\w])[a-z]{3,}(?![-\w])", full_text.lower()))

def fix_hyphens(text, vocab):
    def repl(m):
        a, b = m.group(1), m.group(2)
        if f"{a}-{b}".lower() in KEEP_HYPHEN:
            return m.group(0)
        return a + b if (a + b).lower() in vocab else m.group(0)
    return re.sub(r"\b([A-Za-z]{2,})-([A-Za-z]{2,})\b", repl, text)

In [17]:
# import re
from collections import Counter

KEEP_HYPHEN = {"co-operative", "co-ordination", "up-to"}
_WORD = re.compile(r"(?<![-\w])[A-Za-z]{3,}(?![-\w])")
_SPLIT = re.compile(r"\b([A-Za-z]{2,})-([A-Za-z]{2,})\b")

def build_vocab(full_text: str) -> set[str]:
    """Standalone words (not touching a hyphen) from the WHOLE document."""
    return {w.lower() for w in _WORD.findall(full_text)}

def fix_hyphens(text: str, vocab: set[str], log: Counter | None = None) -> str:
    def repl(m):
        a, b = m.group(1), m.group(2)
        pair = f"{a}-{b}".lower()
        if pair in KEEP_HYPHEN or (a + b).lower() not in vocab:
            return m.group(0)
        if log is not None:
            log[pair] += 1
        return a + b
    return _SPLIT.sub(repl, text)